# 一次學習的超短暖身

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/00-warmup/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""A scalar update with a hand-computable answer; CPU only."""
import torch


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    model = torch.nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        model.weight.fill_(1.0)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    x, target = torch.tensor([[2.0]]), torch.tensor([[4.0]])
    model.train()
    optimizer.zero_grad(set_to_none=True)
    prediction = model(x)
    loss = ((prediction - target) ** 2).mean()
    loss.backward()
    gradient = model.weight.grad.item()
    before = model.weight.item()
    optimizer.step()
    after = model.weight.item()
    model.eval()
    with torch.no_grad():
        new_prediction = model(x)
        new_loss = ((new_prediction - target) ** 2).mean().item()
    print(f"x_shape={tuple(x.shape)}, weight_shape={tuple(model.weight.shape)}")
    print(f"prediction={prediction.item():.2f}, loss={loss.item():.2f}, gradient={gradient:.2f}")
    print(f"weight: {before:.2f} -> {after:.2f}; "
          f"new_prediction={new_prediction.item():.2f}; new_loss={new_loss:.2f}")
    assert abs(gradient + 8.0) < 1e-6
    assert abs(after - 1.8) < 1e-6 and abs(new_loss - 0.16) < 1e-5
    assert abs(new_prediction.item() - 3.6) < 1e-5

    # eval() changes layer behavior; it does not disable autograd.
    assert model.training is False
    assert model(x).requires_grad
    # A newly created model needs an optimizer that references its parameters.
    replacement = torch.nn.Linear(1, 1, bias=False)
    new_optimizer = torch.optim.SGD(replacement.parameters(), lr=0.1)
    assert new_optimizer.param_groups[0]["params"][0] is replacement.weight
    print("eval still tracks gradients; replacement optimizer points to replacement model")


if __name__ == "__main__":
    main()


x_shape=(1, 1), weight_shape=(1, 1)
prediction=2.00, loss=4.00, gradient=-8.00
weight: 1.00 -> 1.80; new_prediction=3.60; new_loss=0.16
eval still tracks gradients; replacement optimizer points to replacement model
